In [1]:
import pandas as pd
import numpy as np
from pathlib import PureWindowsPath
from pathlib import Path

### Data Merging

- Use destination folder and new file to extract the annotations
- The original file dir is used after \NKI to extract the dicom headers
- If there are 2 segmentations use the second

In [2]:
selected = pd.read_csv('/projects/net_contrast_classification/contrast_phase/data/original_data/selected_scans.csv')
processed = pd.read_csv('/projects/net_contrast_classification/contrast_phase/data/original_data/dataset-registration-artinet.csv')
annot = pd.read_csv('/projects/net_contrast_classification/contrast_phase/data/original_data/annotations.csv', )
dicoms = pd.read_csv('/projects/net_contrast_classification/contrast_phase/data/original_data/dicom_metadata.csv')

display(selected.head(),
        annot.head(),
        dicoms.head(),
        processed.head())

,Original File,New File,Destination Folder
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,NET_0000_0000.nii.gz,DICOM-batch1-kalina01
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,NET_0001_0000.nii.gz,DICOM-batch1-kalina01
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,NET_0002_0000.nii.gz,DICOM-batch1-kalina01
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,NET_0003_0000.nii.gz,DICOM-batch1-kalina01
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,NET_0004_0000.nii.gz,DICOM-batch1-kalina01


,file,is_liver_imaged,contrast,phase_timing,no_lesions,ai_segmentation_quality,manual_segmentation_confidence,adjustment_segmentation_difficulty,high_signal_to_noise,metal_artifacts,patient_motion,other,comment,time-stamp,folder
0,NET_0000_0000.nii.gz,2.0,1,1.0,NaN,4.0,4.0,1.0,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
1,NET_0001_0000.nii.gz,NaN,NO_CONTRAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
2,NET_0002_0000.nii.gz,1.0,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
3,NET_0003_0000.nii.gz,1.0,1,2.0,True,NaN,NaN,NaN,True,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
4,NET_0004_0000.nii.gz,1.0,2,2.0,NaN,2.0,3.0,2.0,False,False,False,NaN,NaN,1.706177e+09,DICOM-batch1-kalina01


,Original File,Modality,ImagingFrequency,Manufacturer,ManufacturersModelName,DeviceSerialNumber,PatientPosition,SoftwareVersions,SeriesDescription,ProtocolName,...,ExposureTime,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness,Units
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,CT,0,Toshiba,Aquilion,3936455.0,FFS,V4.51ER010,ABDOMEN A 1.0 FC02,ABDOMEN A 1.0 FC02,...,0.5,150.0,75.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN,NaN
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,CT,0,Toshiba,Aquilion,3936455.0,FFS,V4.51ER010,LEVER -C 1.0 FC03,LEVER -C 1.0 FC03,...,0.5,76.0,38.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN,NaN
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,CT,0,Toshiba,Aquilion,3936455.0,FFS,V4.51ER010,ABDOMEN + C 1.0 FC01,ABDOMEN + C 1.0 FC01,...,0.5,75.0,37.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN,NaN
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,CT,0,Siemens,Sensation Open,662098617.0,FFS,syngo CT 2007S,Maag.abdomen 1.5 B30f,Maag.abdomen 1.5 B30f,...,0.5,168.0,84.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN,NaN
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,CT,0,Siemens,Sensation Open,662098617.0,FFS,syngo CT 2007S,Abdomen V 1.5 B25f,Abdomen V 1.5 B25f,...,0.5,119.0,74.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN,NaN


,SubjectKeyRadiology,ExamDate,SegmentationBatch,NiiFile,SegNiiFile,file,is_liver_imaged,contrast,phase_timing,is_lesionfree,...,AcquisitionNumber,ConvolutionKernel,ExposureTime,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage
0,NKI-d23231-00-0063,2012-04-16,kalina01_0000,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,NET_0000_0000.nii.gz,Partially,Arterial,Too Early,No,...,6.0,FC02,0.5,150,75,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN
1,NKI-d23231-00-0036,2014-02-18,kalina01_0002,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,NET_0002_0000.nii.gz,Yes,Portal,Just Right,Yes,...,7.0,FC01,0.5,75,37,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN
2,NKI-d23231-00-0071,2013-09-24,kalina01_0003,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,NET_0003_0000.nii.gz,Yes,Arterial,Just Right,Yes,...,3.0,B30f,0.5,168,84,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN
3,NKI-d23231-00-0077,2014-12-22,kalina01_0004,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,NET_0004_0000.nii.gz,Yes,Portal,Just Right,No,...,14.0,B25f,0.5,119,74,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN
4,NKI-d23231-00-0049,2009-02-12,kalina01_0005,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...,NET_0005_0000.nii.gz,Yes,Portal,Just Right,Yes,...,14.0,B25f,0.5,280,140,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN


In [3]:
annot[annot[['file', 'folder']].duplicated(keep=False)].sort_values(['folder', 'file'])

,file,is_liver_imaged,contrast,phase_timing,no_lesions,ai_segmentation_quality,manual_segmentation_confidence,adjustment_segmentation_difficulty,high_signal_to_noise,metal_artifacts,patient_motion,other,comment,time-stamp,folder


In [4]:
print(selected.shape, annot.shape, dicoms.shape, processed.shape)

(9012, 3) (8749, 15) (9012, 26) (5572, 42)


In [5]:
selected['SubjectKeyRadiology'] = selected['Original File'].apply(lambda x: PureWindowsPath(x).parts[2])
selected['ExamDate'] = selected['Original File'].apply(lambda x: pd.to_datetime(PureWindowsPath(x).parts[3].split(' ')[0]))
# selected['ProtocolName'] = selected['Original File'].apply(lambda x: PureWindowsPath(x).parts[4].split('.nii')[0])

display(selected.head())

,Original File,New File,Destination Folder,SubjectKeyRadiology,ExamDate
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,NET_0000_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0063,2012-04-16
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,NET_0001_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0070,2015-08-21
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,NET_0002_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0036,2014-02-18
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,NET_0003_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0071,2013-09-24
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,NET_0004_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0077,2014-12-22


#### Find the 9012-8749 = 263 entries in processed

In [6]:
selected_annot = selected.merge(annot, left_on=['Destination Folder','New File'], right_on=['folder', 'file'], how='left')
display(selected_annot.head())

,Original File,New File,Destination Folder,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,phase_timing,no_lesions,ai_segmentation_quality,manual_segmentation_confidence,adjustment_segmentation_difficulty,high_signal_to_noise,metal_artifacts,patient_motion,other,comment,time-stamp,folder
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,NET_0000_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0063,2012-04-16,NET_0000_0000.nii.gz,2.0,1,1.0,NaN,4.0,4.0,1.0,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,NET_0001_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0070,2015-08-21,NET_0001_0000.nii.gz,NaN,NO_CONTRAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,NET_0002_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0036,2014-02-18,NET_0002_0000.nii.gz,1.0,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,NET_0003_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0071,2013-09-24,NET_0003_0000.nii.gz,1.0,1,2.0,True,NaN,NaN,NaN,True,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,NET_0004_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0077,2014-12-22,NET_0004_0000.nii.gz,1.0,2,2.0,NaN,2.0,3.0,2.0,False,False,False,NaN,NaN,1.706177e+09,DICOM-batch1-kalina01


In [7]:
missing = selected[selected_annot.file.isna()]
display(missing.shape, missing.columns)

(263, 5)

Index(['Original File', 'New File', 'Destination Folder',
       'SubjectKeyRadiology', 'ExamDate'],
      dtype='str')

In [8]:
pairs = set(zip(
    missing['SubjectKeyRadiology'],
    missing['ExamDate']
))

exists_mask = list(
    zip(processed['SubjectKeyRadiology'], processed['ExamDate']))


present = processed[ [p in pairs for p in exists_mask] ]
display(missing.head(), present.head())     # => not present in the processed, just not annotated at all

,Original File,New File,Destination Folder,SubjectKeyRadiology,ExamDate
20,../DICOM-batch1\NKI-d23231-00-0076\20070622 Ab...,NET_0020_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0076,2007-06-22
43,../DICOM-batch1\NKI-d23231-00-0075\20080603 Ab...,NET_0043_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0075,2008-06-03
80,../DICOM-batch1\NKI-d23231-00-0048\20190329 CT...,NET_0020_0000.nii.gz,DICOM-batch1-kalina02,NKI-d23231-00-0048,2019-03-29
162,../DICOM-batch1\NKI-d23231-00-0042\20041117 Ab...,NET_0042_0000.nii.gz,DICOM-batch1-kalina03,NKI-d23231-00-0042,2004-11-17
185,../DICOM-batch1\NKI-d23231-00-0051\20061221 Ab...,NET_0005_0000.nii.gz,DICOM-batch1-kalina04,NKI-d23231-00-0051,2006-12-21


,SubjectKeyRadiology,ExamDate,SegmentationBatch,NiiFile,SegNiiFile,file,is_liver_imaged,contrast,phase_timing,is_lesionfree,...,AcquisitionNumber,ConvolutionKernel,ExposureTime,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage


In [9]:
missing.info()

<class 'pandas.DataFrame'>
Index: 263 entries, 20 to 8838
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Original File        263 non-null    str           
 1   New File             263 non-null    str           
 2   Destination Folder   263 non-null    str           
 3   SubjectKeyRadiology  263 non-null    str           
 4   ExamDate             263 non-null    datetime64[us]
dtypes: datetime64[us](1), str(4)
memory usage: 51.7 KB


#### Merge

In [10]:
selected_dicoms = selected.merge(dicoms, on=['Original File'], how='left')
data = selected_dicoms.merge(annot, left_on=['Destination Folder','New File'], right_on=['folder', 'file'], how='right')
data.head()

,Original File,New File,Destination Folder,SubjectKeyRadiology,ExamDate,Modality,ImagingFrequency,Manufacturer,ManufacturersModelName,DeviceSerialNumber,...,ai_segmentation_quality,manual_segmentation_confidence,adjustment_segmentation_difficulty,high_signal_to_noise,metal_artifacts,patient_motion,other,comment,time-stamp,folder
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,NET_0000_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0063,2012-04-16,CT,0,Toshiba,Aquilion,3936455.0,...,4.0,4.0,1.0,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,NET_0001_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0070,2015-08-21,CT,0,Toshiba,Aquilion,3936455.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,NET_0002_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0036,2014-02-18,CT,0,Toshiba,Aquilion,3936455.0,...,NaN,NaN,NaN,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,NET_0003_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0071,2013-09-24,CT,0,Siemens,Sensation Open,662098617.0,...,NaN,NaN,NaN,True,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,NET_0004_0000.nii.gz,DICOM-batch1-kalina01,NKI-d23231-00-0077,2014-12-22,CT,0,Siemens,Sensation Open,662098617.0,...,2.0,3.0,2.0,False,False,False,NaN,NaN,1.706177e+09,DICOM-batch1-kalina01


In [11]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8749 entries, 0 to 8748
Data columns (total 45 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Original File                       8749 non-null   str           
 1   New File                            8749 non-null   str           
 2   Destination Folder                  8749 non-null   str           
 3   SubjectKeyRadiology                 8749 non-null   str           
 4   ExamDate                            8749 non-null   datetime64[us]
 5   Modality                            8749 non-null   str           
 6   ImagingFrequency                    8749 non-null   int64         
 7   Manufacturer                        8749 non-null   str           
 8   ManufacturersModelName              8749 non-null   str           
 9   DeviceSerialNumber                  8348 non-null   float64       
 10  PatientPosition                    

In [12]:
cols = processed.columns.tolist()
cols.extend(['SliceThickness'])
cols.insert(0, 'Original File')
cols.insert(1, 'Destination Folder')
cols.insert(2, 'New File')
cols = [col for col in cols if col not in ['SegmentationBatch', 'NiiFile', 'SegNiiFile']]
cols


['Original File',
 'Destination Folder',
 'New File',
 'SubjectKeyRadiology',
 'ExamDate',
 'file',
 'is_liver_imaged',
 'contrast',
 'phase_timing',
 'is_lesionfree',
 'ai_segmentation_quality',
 'manual_segmentation_confidence',
 'adjustment_segmentation_difficulty',
 'high_signal_to_noise',
 'metal_artifacts',
 'patient_motion',
 'other',
 'comment',
 'time-stamp',
 'Modality',
 'ImagingFrequency',
 'Manufacturer',
 'ManufacturersModelName',
 'DeviceSerialNumber',
 'PatientPosition',
 'SoftwareVersions',
 'SeriesDescription',
 'ProtocolName',
 'ScanOptions',
 'ImageType',
 'SeriesNumber',
 'AcquisitionTime',
 'AcquisitionNumber',
 'ConvolutionKernel',
 'ExposureTime',
 'XRayTubeCurrent',
 'XRayExposure',
 'ImageOrientationPatientDICOM',
 'ConversionSoftware',
 'ConversionSoftwareVersion',
 'BodyPartExamined',
 'RawImage',
 'SliceThickness']

In [13]:
data = data.rename(columns={'no_lesions': 'is_lesionfree'})
data = data[cols]
data.head()

,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,phase_timing,is_lesionfree,...,ConvolutionKernel,ExposureTime,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,DICOM-batch1-kalina01,NET_0000_0000.nii.gz,NKI-d23231-00-0063,2012-04-16,NET_0000_0000.nii.gz,2.0,1,1.0,NaN,...,FC02,0.5,150.0,75.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,DICOM-batch1-kalina01,NET_0001_0000.nii.gz,NKI-d23231-00-0070,2015-08-21,NET_0001_0000.nii.gz,NaN,NO_CONTRAST,NaN,NaN,...,FC03,0.5,76.0,38.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,DICOM-batch1-kalina01,NET_0002_0000.nii.gz,NKI-d23231-00-0036,2014-02-18,NET_0002_0000.nii.gz,1.0,2,2.0,True,...,FC01,0.5,75.0,37.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,DICOM-batch1-kalina01,NET_0003_0000.nii.gz,NKI-d23231-00-0071,2013-09-24,NET_0003_0000.nii.gz,1.0,1,2.0,True,...,B30f,0.5,168.0,84.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,DICOM-batch1-kalina01,NET_0004_0000.nii.gz,NKI-d23231-00-0077,2014-12-22,NET_0004_0000.nii.gz,1.0,2,2.0,NaN,...,B25f,0.5,119.0,74.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN


In [14]:
data['ProtocolName'].unique().tolist()

['ABDOMEN A 1.0 FC02',
 'LEVER -C 1.0 FC03',
 'ABDOMEN + C 1.0 FC01',
 'Maag.abdomen  1.5  B30f',
 'Abdomen V  1.5  B25f',
 'ABDOMEN A 1.0 FC01',
 'Abdomen A  3.0  B31f',
 'Body 1.0 CE   Vol.',
 'Lever -C  1.5  B25f',
 'LEVER -C 1.0 FC01',
 'LEVER -C 1.0 FC07',
 'Mediastinum 2mm',
 'OMC PANCREAS VEN',
 'MPR MEDIASTINUM COR',
 'Abdomen A  1.5  B25f',
 'ABDOMEN + C 1.0 FC02',
 'Abdomen -C  3.0  B31f',
 'Abd. PV 3.0',
 'Abdomen  1.0  I26f  2',
 'Lever -C 2mm',
 'Lever 1.0    Vol.',
 'Lever A  1.5  B25f',
 'Abdomen  1.5  B25f',
 'Lever blanco  1.5  B25f',
 'ThxAbd 1 serie  1.5  B25f',
 'Abd-Contrast 3.0 B30f',
 'Abd-Contrast  3.0  SPO  cor',
 'Abdomen  1.0  Bv38  2',
 'Lever -C  3.0  B31f',
 'Abdomen  3.0  B31f',
 'Abdomen A 2mm',
 'Veneus 3.0 B30f',
 'LEVER A 1.0 FC01',
 'Thorax  1.5  B45f',
 'Abdomen V 2mm',
 'AbdVeneus  3.0  I40f  2',
 'Maag.abdomen  1.5  B25f',
 'Abdomen V  1.5  B31f',
 'Abd laat-art  1.0  Bv38  2',
 'Abdomen maag  1.0  Bv38  2',
 'Abdomen +C  1.5  B30f',
 'lever -C 2m

In [15]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8749 entries, 0 to 8748
Data columns (total 43 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Original File                       8749 non-null   str           
 1   Destination Folder                  8749 non-null   str           
 2   New File                            8749 non-null   str           
 3   SubjectKeyRadiology                 8749 non-null   str           
 4   ExamDate                            8749 non-null   datetime64[us]
 5   file                                8749 non-null   str           
 6   is_liver_imaged                     6652 non-null   float64       
 7   contrast                            8749 non-null   str           
 8   phase_timing                        6652 non-null   float64       
 9   is_lesionfree                       2151 non-null   object        
 10  ai_segmentation_quality            

In [16]:
data.to_csv("/projects/net_contrast_classification/contrast_phase/data/full_data.csv", index=False)

### Mapping

In [17]:
# data = data.dropna(subset=['contrast'])
display(data.contrast.value_counts(), data.contrast.isna().sum(),
        data.phase_timing.value_counts(), data.phase_timing.isna().sum(),
        data.is_liver_imaged.value_counts(), data.is_liver_imaged.isna().sum(),
        data.is_lesionfree.value_counts(), data.is_lesionfree.isna().sum())


contrast
2              3708
1              2488
NO_CONTRAST    2097
0               441
3                15
Name: count, dtype: int64

np.int64(0)

phase_timing
2.0    5115
1.0     916
0.0     438
3.0     183
Name: count, dtype: int64

np.int64(2097)

is_liver_imaged
1.0    5892
0.0     381
2.0     379
Name: count, dtype: int64

np.int64(2097)

is_lesionfree
True    2151
Name: count, dtype: int64

np.int64(6598)

In [18]:
data[(data.contrast == "NO_CONTRAST") & (data.phase_timing.notna())]       # all NC images don't have phase_timing (phase_timing = NaN)

,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,phase_timing,is_lesionfree,...,ConvolutionKernel,ExposureTime,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness


In [19]:
data[(data.contrast == "NO_CONTRAST") & (data.is_liver_imaged.notna())]       # all NC images don't have liver ???

,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,phase_timing,is_lesionfree,...,ConvolutionKernel,ExposureTime,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness


In [20]:
contrast_mapping = {"1": "Arterial", "2": "Portal", "NO_CONTRAST": "Non-contrast", "3": "Late Phase", "0": "0"}  # contrast = 0 (forgot to select) or NA (no liver - skip, ignore from dataset)
time_mapping = {1.0: "Too Early", 2.0: "Just Right", 3.0: "Too Late", 0.0: 0.0}                                  # phase_timing = NA ??? (skip, ignore from dataset)
liver_mapping = {1.0: "Yes", 2.0: "Partially", 0.0: "0"}                                                         # is_liver_imaged = "0" or NA ???
lesion_mapping = {True: "Yes", None: "No"}


data['contrast'] = data['contrast'].map(contrast_mapping)
data['phase_timing'] = data['phase_timing'].map(time_mapping)
data['is_liver_imaged'] = data['is_liver_imaged'].map(liver_mapping)
data.loc[data['file'].notna(), 'is_lesionfree'] = data.loc[data['file'].notna(), 'is_lesionfree'].map(lesion_mapping).fillna("No")

# If contrast == 0 and ProtocolName contains -c => contrast = NC
mask_nc = (
    # data['contrast'].eq("0") &
    data['ProtocolName'].str.contains(r'-c(?=\W|$)', case=False, na=False) # -C before anything that is not a letter or number
)

data.loc[mask_nc, 'contrast'] = 'Non-contrast'


# If contrast == NC => phase_timing = NC
mask = data['contrast'].eq('Non-contrast')
data.loc[mask, 'phase_timing'] = 'Non-contrast'


# If is_liver_imaged is 0 or NA and contrast is not NA and ProtocolName contains lever => is_liver_imaged = Yes

mask_liver = (
    (data['is_liver_imaged'].eq("0") | data['is_liver_imaged'].isna()) &
    data['ProtocolName'].str.lower().str.match(r'^lever\b', case=False,na=False)
)

data.loc[mask_liver, 'is_liver_imaged'] = 'Yes'

In [21]:
# annot = annot.dropna(subset=['contrast'])
display(data.contrast.value_counts(), f"Null values: {int(data.contrast.isna().sum())}",
        data.is_liver_imaged.value_counts(), f"Null values: {int(data.is_liver_imaged.isna().sum())}",
        data.phase_timing.value_counts(), f"Null values: {int(data.phase_timing.isna().sum())}",
        data.is_lesionfree.value_counts(), f"Null values: {int(data.is_lesionfree.isna().sum())}")


contrast
Portal          3706
Arterial        2487
Non-contrast    2122
0                419
Late Phase        15
Name: count, dtype: int64

'Null values: 0'

is_liver_imaged
Yes          7879
Partially     379
0             330
Name: count, dtype: int64

'Null values: 161'

phase_timing
Just Right      5112
Non-contrast    2122
Too Early        916
0.0              416
Too Late         183
Name: count, dtype: int64

'Null values: 0'

is_lesionfree
No     6598
Yes    2151
Name: count, dtype: int64

'Null values: 0'

In [22]:
unknown_contrast_timing_or_liver = data[(data.contrast == "0") 
                                                 | (data.is_liver_imaged == "0")
                                                 | (data.phase_timing == 0.0)
              ]

unknown_contrast_timing_or_liver.to_csv("/projects/net_contrast_classification/contrast_phase/data/unknown_contrast_timing_or_liver.csv", index=False)
unknown_contrast_timing_or_liver

,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,phase_timing,is_lesionfree,...,ConvolutionKernel,ExposureTime,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness
811,../DICOM-batch1\NKI-d23231-00-0001\20070815 CT...,DICOM-batch1-kalina99,NET_0001_0000.nii.gz,NKI-d23231-00-0001,2007-08-15,NET_0001_0000.nii.gz,0,Portal,Just Right,No,...,B,NaN,235.0,179.0,"[1, 0, 0, 0, 0, -1]",dcm2niix,v1.0.20230411,NaN,False,NaN
865,../DICOM-batch1\NKI-d23231-00-0036\20060814 Ab...,DICOM-batch1-kalina99,NET_0057_0000.nii.gz,NKI-d23231-00-0036,2006-08-14,NET_0057_0000.nii.gz,0,Portal,Just Right,Yes,...,B30f,0.5,137.0,68.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN
1490,../DICOM-batch2\NKI-d23231-00-0146\20181029 CT...,DICOM-batch2-david01,NET_0063_0000.nii.gz,NKI-d23231-00-0146,2018-10-29,NET_0063_0000.nii.gz,0,0,0.0,No,...,FC01,0.5,150.0,75.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN
1508,../DICOM-batch2\NKI-d23231-00-0182\20130726 CT...,DICOM-batch2-david01,NET_0081_0000.nii.gz,NKI-d23231-00-0182,2013-07-26,NET_0081_0000.nii.gz,0,Arterial,Just Right,No,...,FC01,0.5,95.0,47.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN
1643,../DICOM-batch2\NKI-d23231-00-0128\20041118 Ab...,DICOM-batch2-giovanni01,NET_0014_0000.nii.gz,NKI-d23231-00-0128,2004-11-18,NET_0014_0000.nii.gz,0,0,0.0,No,...,B30f,0.5,141.0,58.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8721,../DICOM-batch3\NKI-d23231-00-0885\20210421 Re...,DICOM-batch3-kalina99,NET_5804_0000.nii.gz,NKI-d23231-00-0885,2021-04-21,NET_5804_0000.nii.gz,0,0,0.0,No,...,FC08,0.5,100.0,50.0,"[0, 1, 0, 0, 0, -1]",dcm2niix,v1.0.20230411,NaN,False,NaN
8726,../DICOM-batch3\NKI-d23231-00-0887\20210318 Re...,DICOM-batch3-kalina99,NET_5809_0000.nii.gz,NKI-d23231-00-0887,2021-03-18,NET_5809_0000.nii.gz,0,0,0.0,No,...,Bf37f\2,0.5,259.0,143.0,"[0.999451, -0.0271001, 0.0190742, 0.0190672, -...",dcm2niix,v1.0.20230411,CT THORAX EN ___,False,NaN
8727,../DICOM-batch3\NKI-d23231-00-0887\20210318 Re...,DICOM-batch3-kalina99,NET_5810_0000.nii.gz,NKI-d23231-00-0887,2021-03-18,NET_5810_0000.nii.gz,0,0,0.0,No,...,Bf37f\2,0.5,259.0,143.0,"[0.0271051, 0.999633, 0, 0.0190672, -0.0005170...",dcm2niix,v1.0.20230411,CT THORAX EN ___,False,NaN
8741,../DICOM-batch3\NKI-d23231-00-0888\20210218 Re...,DICOM-batch3-kalina99,NET_5824_0000.nii.gz,NKI-d23231-00-0888,2021-02-18,NET_5824_0000.nii.gz,0,0,0.0,No,...,I40f\2,0.5,202.0,168.0,"[1, 0, 0, 0, 0, -1]",dcm2niix,v1.0.20230411,ABDOMEN,False,NaN


In [23]:
display(data[data.is_liver_imaged.isna()]['contrast'].value_counts(),
        data[data.is_liver_imaged.isna()]['ProtocolName'].value_counts())



contrast
Non-contrast    161
Name: count, dtype: int64

ProtocolName
1.0 FC03                    25
Abdomen  1.5  B30f          13
ABDOMEN - C 1.0 FC02        10
ABDOMEN - C 1.0 FC01        10
Abdomen  1.5  B25f           8
                            ..
Abdomen   1.5  B25f          1
IVP (blanco)  3.0  B30s      1
Abdomen blanc  1.5  B25f     1
Abdomen                      1
Thorax  3.0  I31f  2         1
Name: count, Length: 62, dtype: int64

In [24]:
missing_liver_NC = data[data.is_liver_imaged.isna()]
missing_liver_NC.to_csv("/projects/net_contrast_classification/contrast_phase/data/missing_liver_NC.csv", index=False)

In [25]:
data[data.phase_timing == 0.0]['contrast'].value_counts()

contrast
0           410
Arterial      5
Portal        1
Name: count, dtype: int64

In [26]:
print("Entries with phase_timing == 0:", len(data[data.phase_timing == 0.0]))
data[(data.phase_timing == 0.0) & (data['contrast'].notna())][
    ["file", "ProtocolName", "contrast", "phase_timing", "is_liver_imaged"]]

Entries with phase_timing == 0: 416


,file,ProtocolName,contrast,phase_timing,is_liver_imaged
1490,NET_0063_0000.nii.gz,Body 1.0 CE Vol.,0,0.0,0
1643,NET_0014_0000.nii.gz,Abdomen BB 1.5 B30f,0,0.0,0
1666,NET_0037_0000.nii.gz,Abdomen 1.5 B30f,0,0.0,0
1674,NET_0045_0000.nii.gz,Abdomen OB 1.5 B30f,0,0.0,0
1691,NET_0062_0000.nii.gz,Lever 1.0 Vol.,0,0.0,Yes
...,...,...,...,...,...
8721,NET_5804_0000.nii.gz,SOFT TISSUE 3.000 CE,0,0.0,0
8726,NET_5809_0000.nii.gz,ThoraxAbd cor. 3.0 MPR cor,0,0.0,0
8727,NET_5810_0000.nii.gz,ThoraxAbd sag.. 3.0 MPR sag,0,0.0,0
8741,NET_5824_0000.nii.gz,Bobuik art 1.0 MPR cor,0,0.0,0


In [27]:
print("Entries with contrast == 0:", len(data[data.contrast == '0']))
data[data.contrast == '0'][['SubjectKeyRadiology', 'ExamDate', 'contrast', "phase_timing", 'ProtocolName', 'is_liver_imaged']]

Entries with contrast == 0: 419


,SubjectKeyRadiology,ExamDate,contrast,phase_timing,ProtocolName,is_liver_imaged
1490,NKI-d23231-00-0146,2018-10-29,0,0.0,Body 1.0 CE Vol.,0
1643,NKI-d23231-00-0128,2004-11-18,0,0.0,Abdomen BB 1.5 B30f,0
1666,NKI-d23231-00-0102,2006-03-02,0,0.0,Abdomen 1.5 B30f,0
1674,NKI-d23231-00-0162,2005-02-14,0,0.0,Abdomen OB 1.5 B30f,0
1691,NKI-d23231-00-0164,2017-10-02,0,0.0,Lever 1.0 Vol.,Yes
...,...,...,...,...,...,...
8721,NKI-d23231-00-0885,2021-04-21,0,0.0,SOFT TISSUE 3.000 CE,0
8726,NKI-d23231-00-0887,2021-03-18,0,0.0,ThoraxAbd cor. 3.0 MPR cor,0
8727,NKI-d23231-00-0887,2021-03-18,0,0.0,ThoraxAbd sag.. 3.0 MPR sag,0
8741,NKI-d23231-00-0888,2021-02-18,0,0.0,Bobuik art 1.0 MPR cor,0


In [35]:
# print("Entries with contrast = null :", len(data[data.contrast.isna()]))
# missing_entries = data[data.contrast.isna()][['Original File', 'Destination Folder', 'file', 'SubjectKeyRadiology', 'ExamDate', 'contrast', "phase_timing", 'ProtocolName', 'is_liver_imaged']]
# missing_entries.to_csv("data/missing_entries.csv", index=False)
# missing_entries.head()

Takeaways: 
- all entries that received a missing value for contrast => missing annotation file
- For 424 entries we have a is_liver_imaged = NaN
- Out of those, 263 entries are present in selected_scans.csv and not present in the annotated data
- The other 161 have contrast = NC

In [28]:
# counts
sel_counts = selected['Destination Folder'].value_counts()
ann_counts = annot['folder'].value_counts()

# positive surplus per folder
surplus = sel_counts.sub(ann_counts, fill_value=0).astype(int)
surplus = surplus[surplus > 0]   # folder -> number of extra rows

# pick the extra rows from selected (takes last N rows per folder)
surplus_rows = (
    annot[annot['folder'].isin(surplus.index)]
    .groupby('folder', group_keys=False)
    .apply(lambda g: g.tail(surplus[g.name]))
)

display(surplus_rows)


,file,is_liver_imaged,contrast,phase_timing,no_lesions,ai_segmentation_quality,manual_segmentation_confidence,adjustment_segmentation_difficulty,high_signal_to_noise,metal_artifacts,patient_motion,other,comment,time-stamp
56,NET_0058_0000.nii.gz,1.0,2,2.0,True,NaN,NaN,NaN,True,False,False,NaN,NaN,1.706189e+09
57,NET_0059_0000.nii.gz,1.0,2,2.0,NaN,4.0,4.0,2.0,False,False,False,NaN,NaN,1.706190e+09
116,NET_0059_0000.nii.gz,1.0,1,1.0,NaN,3.0,2.0,4.0,True,False,False,NaN,The lesions are extremely hard to see and segm...,1.706277e+09
175,NET_0059_0000.nii.gz,2.0,1,1.0,True,NaN,NaN,NaN,True,False,False,NaN,post RFA-changes,1.706534e+09
232,NET_0058_0000.nii.gz,1.0,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.706545e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8744,NET_5827_0000.nii.gz,1.0,2,2.0,NaN,5.0,4.0,1.0,False,False,False,NaN,NaN,1.728478e+09
8745,NET_5828_0000.nii.gz,1.0,1,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.728478e+09
8746,NET_5829_0000.nii.gz,1.0,2,2.0,NaN,5.0,3.0,1.0,False,False,False,NaN,NaN,1.720620e+09
8747,NET_5830_0000.nii.gz,1.0,1,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.720001e+09


TO DO:
- Ask why do we have duplicates - checked
- Ask what is the mapping of contrast, phase timing, etc.

### Reformatting

In [29]:
display(data.AcquisitionTime.head())
data['AcquisitionTime_sec'] = pd.to_timedelta(data['AcquisitionTime']).dt.total_seconds()
display(data.AcquisitionTime_sec.head())

data["ExamDate"] = pd.to_datetime(data["ExamDate"])

0    11:10:31.700000
1    08:34:13.900000
2    09:03:19.600000
3    12:30:13.306920
4    14:19:47.104180
Name: AcquisitionTime, dtype: str

0    40231.70000
1    30853.90000
2    32599.60000
3    45013.30692
4    51587.10418
Name: AcquisitionTime_sec, dtype: float64

In [30]:
data.BodyPartExamined.value_counts()

data[data.BodyPartExamined.isna()]['ProtocolName'].value_counts()

mask = data['BodyPartExamined'].isna()

# extract first word from ProtocolName
first_word = (
    data.loc[mask, 'ProtocolName']
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
    .str.upper()
)

# allowed mappings
mapping = {
    'PANCREAS': 'PANCREAS',
    'SOFT': 'SOFT TISSUE',
    'ABDOMEN': 'ABDOMEN',
    'ABDOMEN+': 'ABDOMEN',
    'Abdomen': 'ABDOMEN',
    'Abd.': 'ABDOMEN',
    'Lever': 'LEVER',
    'LEVER': 'LEVER',
    'lever': 'LEVER',
    'BODY': 'BODY',
    'Body': 'BODY',
    'Mediastinum': 'MEDIASTINUM',
    'MEDIASTINUM': 'MEDIASTINUM'
}

# fill only if first word is in mapping
data.loc[mask, 'BodyPartExamined'] = first_word.map(mapping)

print("Number of missing BodyPartExamined values:", int(data['BodyPartExamined'].isna().sum()))
data['BodyPartExamined'].value_counts() #ASK KALINA WHICH IMAGES CAN BE IGNORED

Number of missing BodyPartExamined values: 111


BodyPartExamined
ABDOMEN             7585
LEVER                629
CHEST                262
MEDIASTINUM           43
BODY                  38
THORAX                16
BUIK                   8
CT LEVER 3_F           7
CT BBUIK ZMK           5
CT_ABD C_ MET VO       4
PANCREAS               4
CT ABDOMEN MULTI       4
SOFT TISSUE            4
CT HALS_THORA___       3
CT THORAX_ABD___       3
CT THORAX EN ___       3
CT_DUNNE DARM          2
CHEST_ABDOMEN          2
CT ABDOMEN             2
CT_THX_ABD             2
CT ABDOMEN MET I       2
CT ABD C_ MET VB       2
NECK                   2
CT BB_OB_MC_           1
WHOLEBODY              1
CT_ABD C_ZONDER        1
CT CEREBRUM MET        1
_S_297                 1
CHEST _ ABDOMEN        1
Name: count, dtype: int64

In [31]:
# TO DO: check 
display(data.SliceThickness.value_counts())
data[data['SliceThickness'].isin([0.625,0.742188,0.894531,3.0])][['ProtocolName', 'SliceThickness']]

SliceThickness
3.0    1
Name: count, dtype: int64

,ProtocolName,SliceThickness
3276,mpr cor 3 mm,3.0


In [32]:
# Replace commas with dots in numbers
data['ProtocolName'] = data['ProtocolName'].str.replace(r'(\d+),(\d+)', r'\1.\2', regex=True)

print(data.ProtocolName.value_counts())

mask = data['SliceThickness'].isna()

data.loc[mask, 'SliceThickness'] = (
    data.loc[mask, 'ProtocolName']
        .str.extract(r'(\d+\.\d+|\d+(?=\s*[mM]{2}))')[0]
        .astype(float)
)

print(f"\nNumber of missing values in SliceThickness after extraction: {data.SliceThickness.isna().sum()}")

data.SliceThickness.value_counts()

ProtocolName
Body 1.0 CE   Vol.                1365
Abdomen A  1.5  B25f               930
Lever -C  1.5  B25f                820
Abdomen V  1.5  B25f               776
ABDOMEN + C 1.0 FC01               505
                                  ... 
SOFT TISSUE 3.000 CE                 1
ThoraxAbd ax.  3.0  Bf37  2          1
ThoraxAbd cor.  3.0  MPR  cor        1
ThoraxAbd sag..  3.0  MPR  sag       1
Abd PV  1.0  Br36  3                 1
Name: count, Length: 444, dtype: int64

Number of missing values in SliceThickness after extraction: 46


SliceThickness
1.0    4309
1.5    3350
3.0     723
2.0     321
Name: count, dtype: int64

In [33]:
# Late arterial are counted as arterial ASK KALINA

def map_phase(protocol_name):
    protocol_name = str(protocol_name).lower()  # normalize
    if any(k in protocol_name for k in [' a ', ' art', ' art.', 'laat-art', 'arterieel', 'arterial']):
        return 'Arterial'
    elif any(k in protocol_name for k in ['p ', ' pv ',' v ', 'portaal', 'venous', 'veneus', 'ven', 'port', ' vv ']):
        if any(k in protocol_name for k in ['laat veneus']): #
            return 'Late Venous'
        return 'Portal'
    elif any(k in protocol_name for k in ['late fase', 'delayed phase', 'min ', 'min. ']):
        return 'Late Phase'
    elif any(k in protocol_name for k in [' -c ']):
        return 'Non-contrast'
    elif any(k in protocol_name for k in ['uitscheiding']):
        return 'Excretory'
    else:
        return np.nan  # unknown / other

# Apply mapping
data['DICOM_phase'] = data['ProtocolName'].apply(map_phase)

display(data[data.DICOM_phase.isna()].contrast.value_counts())

no_phase = data[data.DICOM_phase.isna()][['ProtocolName', 'DICOM_phase', 'SliceThickness', 'BodyPartExamined']]
print("Number of rows with missing DICOM_phase:", len(no_phase))
print("Number of contrast = NaN rows with missing DICOM_phase:", data[data.DICOM_phase.isna()]['contrast'].isna().sum())


contrast
Portal          2672
Non-contrast     596
Arterial         285
0                276
Late Phase         6
Name: count, dtype: int64

Number of rows with missing DICOM_phase: 3835
Number of contrast = NaN rows with missing DICOM_phase: 0


In [34]:
list(np.unique(no_phase.ProtocolName))

['1 - THOR.+C. 3.0 B30f',
 '1.0 FC01',
 '1.0 FC03',
 '1.0 FC07',
 '2 - ABD.+C. 3.0 B30f',
 '2MM LONGSETTING',
 '2MM MEDIASTINUM',
 'A 30',
 'ABDO  SS IV  3.0  B30f',
 'ABDOMEN + C 1.0 FC01',
 'ABDOMEN + C 1.0 FC02',
 'ABDOMEN - C 1.0 FC01',
 'ABDOMEN - C 1.0 FC02',
 'ABDOMEN 3 MM',
 'ABDOMEN COR',
 'ABDOMEN SAG',
 'ABDOMEN+ th + C 1.0 FC01',
 'ART COR',
 'ART SAG',
 'Abd  ThorAbd  1.0  Br40  3',
 'Abd Routine  1.5  I31f sv ax',
 'Abd Routine  3.0  I31f  3',
 'Abd Routine  3.0  I31f  ax',
 'Abd Routine  3.0  MPR  cor',
 'Abd Routine  3.0  MPR  sag',
 'Abd algemeen  2.0  B31f',
 'Abd laat  2.0  I31f  3',
 'Abd-Contrast  3.0  B40f',
 'Abd-Contrast  3.0  I40f  2',
 'Abd-Contrast  3.0  MPR  cor',
 'Abd-Contrast  3.0  SPO  cor',
 'Abd-Contrast 3.0 B30f',
 'Abd. Blanco  3.0  B30f',
 'Abd. laat  2.0  B30f',
 'Abd.Routine  3.0  B30f',
 'AbdRoutine 3.0 B30s',
 'Abd_Routine  1.5  I31f  3 sv ax',
 'Abd_Routine  3.0  I31f  3 ax',
 'Abd_Routine  3.0  MPR  cor',
 'Abd_Routine  3.0  MPR  sag',
 'Abdom

### Duplicated

In [35]:
data[data[['SubjectKeyRadiology', 'ExamDate', 'file', 'ProtocolName']].duplicated(keep=False)].sort_values(['SubjectKeyRadiology', 'Original File'])


,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,phase_timing,is_lesionfree,...,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness,AcquisitionTime_sec,DICOM_phase


In [36]:
print('Number of patients:', data['SubjectKeyRadiology'].nunique(), 
      '\nNumber of exams:', data['ExamDate'].nunique(), 
      '\nNumber of files:', data['Original File'].nunique(), 
      '\nNumber of patients with both arterial and portal:', data[data.contrast.isin(['Arterial', 'Portal'])]
                                                                                .groupby('SubjectKeyRadiology')['contrast']
                                                                                .nunique()
                                                                                .eq(2)
                                                                                .sum()
                                                                                )
                                                                                

Number of patients: 665 
Number of exams: 2630 
Number of files: 8749 
Number of patients with both arterial and portal: 515


In [37]:
cols = ['SubjectKeyRadiology', 'ExamDate', 'AcquisitionTime', 'AcquisitionTime_sec', 'contrast', 'phase_timing',
         'metal_artifacts', 'ProtocolName', 'ConvolutionKernel', 'time-stamp', 'BodyPartExamined']

contrast_conflicts = (
    data
    .groupby([
        'SubjectKeyRadiology',
        'ExamDate',
        'AcquisitionTime',
        'ProtocolName'
    ])['contrast']
    .nunique()
    .reset_index()
)

# Keep only cases where contrast differs
contrast_conflicts = contrast_conflicts[contrast_conflicts['contrast'] > 1]

print("Cases with different contrast phases due to reconstruction differences:", len(contrast_conflicts))

timing_conflicts = (
    data
    .groupby([
        'SubjectKeyRadiology',
        'ExamDate',
        'AcquisitionTime',
        'ProtocolName'
    ])['phase_timing']
    .nunique()
    .reset_index()
)

# Keep only cases where contrast differs
timing_conflicts = timing_conflicts[timing_conflicts['phase_timing'] > 1]

print("Cases with different phase timings due to reconstruction differences:", len(timing_conflicts))

Cases with different contrast phases due to reconstruction differences: 0
Cases with different phase timings due to reconstruction differences: 0


Same acquisition time, different phase timing due to reconstruction:


In [38]:
display(
    data[data.SubjectKeyRadiology == 'NKI-d23231-00-0888']
    .sort_values(by=['SubjectKeyRadiology', 'ExamDate','AcquisitionTime'],ascending = [True, True, True])[cols].head(2),
    data[data.SubjectKeyRadiology == 'NKI-d23231-00-0723']
    .sort_values(by=['SubjectKeyRadiology', 'ExamDate','AcquisitionTime'],ascending = [True, True, True])[cols].head(2)
)


,SubjectKeyRadiology,ExamDate,AcquisitionTime,AcquisitionTime_sec,contrast,phase_timing,metal_artifacts,ProtocolName,ConvolutionKernel,time-stamp,BodyPartExamined
8740,NKI-d23231-00-0888,2021-02-18,09:14:8.153000,33248.153,Arterial,Too Early,False,Bobuik art 1.0 I40f 2,I40f\2,1.728478e+09,ABDOMEN
8741,NKI-d23231-00-0888,2021-02-18,09:14:8.153000,33248.153,0,0.0,False,Bobuik art 1.0 MPR cor,I40f\2,1.721042e+09,ABDOMEN


,SubjectKeyRadiology,ExamDate,AcquisitionTime,AcquisitionTime_sec,contrast,phase_timing,metal_artifacts,ProtocolName,ConvolutionKernel,time-stamp,BodyPartExamined
7670,NKI-d23231-00-0723,2018-07-10,10:46:32.493000,38792.493,Arterial,Too Late,False,Abd_Routine 1.5 I31f 3 sv ax,I31f\3,1.721649e+09,BUIK
7671,NKI-d23231-00-0723,2018-07-10,10:46:32.493000,38792.493,Arterial,Just Right,False,Abd_Routine 3.0 I31f 3 ax,I31f\3,1.722601e+09,BUIK


In [39]:
dups = data[data[['SubjectKeyRadiology', 'ExamDate', 'AcquisitionTime','ProtocolName']].duplicated()]
diff_kernel = data[data[['SubjectKeyRadiology', 'ExamDate', 'contrast','phase_timing','AcquisitionTime', 'ConvolutionKernel']].duplicated()]

print("Duplicates:", len(dups))
print("Duplicates with different reconstruction:", len(diff_kernel))

Duplicates: 0
Duplicates with different reconstruction: 73


Metal Artifacts

In [40]:
data[data.metal_artifacts == True][['SubjectKeyRadiology', 'ExamDate', 'AcquisitionTime', 'ProtocolName', 'metal_artifacts']]


,SubjectKeyRadiology,ExamDate,AcquisitionTime,ProtocolName,metal_artifacts
7,NKI-d23231-00-0054,2015-04-09,12:30:30.840200,Abdomen A 3.0 B31f,True
44,NKI-d23231-00-0086,2017-01-24,16:34:22.275420,Abdomen V 1.5 B25f,True
121,NKI-d23231-00-0086,2017-08-21,16:00:53.850000,Body 1.0 CE Vol.,True
146,NKI-d23231-00-0093,2016-07-08,17:00:46.700000,Body 1.0 CE Vol.,True
167,NKI-d23231-00-0010,2007-02-13,11:35:12.000000,Abdomen 2mm,True
...,...,...,...,...,...
8160,NKI-d23231-00-0786,2017-12-12,18:59:46.200000,Body 1.0 CE Vol.,True
8162,NKI-d23231-00-0786,2018-04-05,11:31:18.600000,Body 1.0 CE Vol.,True
8247,NKI-d23231-00-0801,2019-04-09,17:17:3.135760,Abdomen A 1.5 B25f,True
8248,NKI-d23231-00-0801,2019-04-09,17:17:52.525400,Abdomen V 1.5 B25f,True


In [41]:

data[data['SubjectKeyRadiology'].isin(diff_kernel['SubjectKeyRadiology']) 
     & (data['ExamDate'].isin(diff_kernel['ExamDate']))
     & (data['metal_artifacts'] == True)
     ].sort_values(by=['SubjectKeyRadiology', 'ExamDate','AcquisitionTime'],ascending = [True, True, True])[cols]

,SubjectKeyRadiology,ExamDate,AcquisitionTime,AcquisitionTime_sec,contrast,phase_timing,metal_artifacts,ProtocolName,ConvolutionKernel,time-stamp,BodyPartExamined
6756,NKI-d23231-00-0535,2019-09-18,11:44:33.167000,42273.16700,Arterial,Just Right,True,Abd laat-art 1.0 I26f 2 iMAR,I26f\2,1.727690e+09,ABDOMEN
6757,NKI-d23231-00-0535,2019-09-18,11:44:33.167000,42273.16700,Arterial,Just Right,True,Abd laat-art 1.0 I26f 2,I26f\2,1.722331e+09,ABDOMEN
6758,NKI-d23231-00-0535,2019-09-18,11:45:11.105000,42311.10500,Portal,Just Right,True,Abdomen 1.0 I26f 2 iMAR,I26f\2,1.727690e+09,ABDOMEN
6759,NKI-d23231-00-0535,2019-09-18,11:45:11.105000,42311.10500,Portal,Just Right,True,Abdomen 1.0 I26f 2,I26f\2,1.721651e+09,ABDOMEN
6765,NKI-d23231-00-0535,2021-03-18,13:50:22.650500,49822.65050,Arterial,Just Right,True,Abdomen A 1.5 B25f,B25f,1.727690e+09,ABDOMEN
6766,NKI-d23231-00-0535,2021-03-18,13:50:57.198290,49857.19829,Portal,Just Right,True,Abdomen V 1.5 B25f,B25f,1.719923e+09,ABDOMEN
6771,NKI-d23231-00-0535,2022-07-19,14:01:30.540000,50490.54000,Arterial,Just Right,True,Abd laat-art 1.0 Bv38 2 iMAR,Bv38f\2,1.719841e+09,ABDOMEN
6772,NKI-d23231-00-0535,2022-07-19,14:01:30.540000,50490.54000,Arterial,Just Right,True,Abd laat-art 1.0 Bv38 2,Bv38f\2,1.720527e+09,ABDOMEN
6773,NKI-d23231-00-0535,2022-07-19,14:02:7.336000,50527.33600,Portal,Just Right,True,Abdomen 1.0 Bf37 2 iMAR,Bf37f\2,1.727694e+09,ABDOMEN
6774,NKI-d23231-00-0535,2022-07-19,14:02:7.336000,50527.33600,Portal,Just Right,True,Abdomen 1.0 Bv38 2,Bv38f\2,1.727694e+09,ABDOMEN


In [42]:
imar_exams = dict(data[data.ProtocolName.str.contains('iMAR', case=False, na=False)][["SubjectKeyRadiology", "ExamDate"]])
imar_exams
imar_data = data[data['SubjectKeyRadiology'].isin(imar_exams['SubjectKeyRadiology'])
     & (data['ExamDate'].isin(imar_exams['ExamDate']))].sort_values(by=['SubjectKeyRadiology', 'ExamDate','AcquisitionTime'],ascending = [True, True, True])

imar_data.to_csv("/projects/net_contrast_classification/contrast_phase/data/imar_exams.csv", index=False)

### Labeling Inconsistensies

In [43]:
# checked = pd.read_csv("checked.csv")
# print(f"Kalina double-checked: {len(checked)} entries")
# checked

# idx = data['NiiFile'].isin(checked['NiiFile'])
# data.loc[idx, 'contrast'] = data.loc[idx, 'NiiFile'].map(
#     checked.set_index('NiiFile')['contrast']
# )

# print("Entries with contrast == 0:", len(data[data.contrast == '0']))
# data[data.contrast == '0'][["NiiFile", 'contrast', 'ProtocolName', 'DICOM_phase']]

In [44]:
print('Number of NC entries according to DICOM:', len(data[data.DICOM_phase == "Non-contrast"]),
      '\nContrast value counts:\n\t', data[data.DICOM_phase == "Non-contrast"]['contrast'].value_counts(),
      '\nSum of contrast values:', data[data.DICOM_phase == "Non-contrast"]['contrast'].value_counts().sum(),
      '\nNumber of missing contrast values:', data[data.DICOM_phase == "Non-contrast"]['contrast'].isna().sum())
# data.loc[data['DICOM_phase'].str.lower().isin(['non-contrast']), 'contrast'] = 'Non-contrast'

Number of NC entries according to DICOM: 1514 
Contrast value counts:
	 contrast
Non-contrast    1514
Name: count, dtype: int64 
Sum of contrast values: 1514 
Number of missing contrast values: 0


In [45]:
diff_dicom = data[(data["DICOM_phase"] != data["contrast"]) 
                  & (data["DICOM_phase"].notna()
                  & data.contrast.notna()) 
                  ][["SubjectKeyRadiology","ExamDate", "ProtocolName", "contrast", "phase_timing", "DICOM_phase", "is_liver_imaged", 'comment', 'other']]
# diff_dicom.to_csv("diff_dicom.csv", index=False)

print("Entries with contrast label different than the DICOM label", len(diff_dicom))
diff_dicom

Entries with contrast label different than the DICOM label 206


,SubjectKeyRadiology,ExamDate,ProtocolName,contrast,phase_timing,DICOM_phase,is_liver_imaged,comment,other
303,NKI-d23231-00-0005,2007-11-16,PORTAAL 3MM.,Late Phase,Just Right,Portal,Yes,NaN,NaN
500,NKI-d23231-00-0007,2010-05-17,Abdomen V 2mm,Arterial,Too Late,Portal,Yes,NaN,NaN
720,NKI-d23231-00-0075,2017-09-12,axiaal 3 mm arterieel,Portal,Just Right,Arterial,Yes,NaN,NaN
809,NKI-d23231-00-0005,2020-01-28,Abdomen A 1.5 B25f,Non-contrast,Non-contrast,Arterial,NaN,NaN,NaN
810,NKI-d23231-00-0001,2007-08-15,COR 2MM ART,Portal,Just Right,Arterial,Yes,NaN,EXCLUDE; coronal reconstruction
...,...,...,...,...,...,...,...,...,...
8638,NKI-d23231-00-0866,2021-11-18,AbdVeneus 3.0 MPR cor,0,0.0,Portal,0,NaN,exclude - coronal
8639,NKI-d23231-00-0866,2021-11-18,AbdVeneus 3.0 MPR sag,0,0.0,Portal,0,NaN,exclude - sagittal
8716,NKI-d23231-00-0882,2022-03-22,Abdomen Art 1.0 Br36 3,Portal,Just Right,Arterial,Yes,lots and lots of cysts,NaN
8741,NKI-d23231-00-0888,2021-02-18,Bobuik art 1.0 MPR cor,0,0.0,Arterial,0,NaN,exclude - coronal


In [46]:
print('Number of entries with contrast = 0 and other field is NA:', data[data["contrast"] == '0'].other.isna().sum())
print('Number of entries with contrast = 0 and other field is not NA:', data[data["contrast"] == '0'].other.notna().sum())

Number of entries with contrast = 0 and other field is NA: 23
Number of entries with contrast = 0 and other field is not NA: 396


In [47]:
diff_dicom.contrast.value_counts()

contrast
0               143
Portal           35
Arterial         12
Non-contrast     12
Late Phase        4
Name: count, dtype: int64

In [48]:
diff_dicom.contrast.value_counts()

# IF CONTRAST = 0 AND DICOM = NAN => UNDETERMINED PHASE
undetermined = data[((data.contrast == '0')) & (data.DICOM_phase.isna()) & (data["is_liver_imaged"]!= '0')][["SubjectKeyRadiology", "ExamDate", "ProtocolName", "contrast", "phase_timing", "DICOM_phase"]]

print('Number of undetermined phases:', len(undetermined))
undetermined[["SubjectKeyRadiology", "ExamDate", "ProtocolName", "contrast", "phase_timing", "DICOM_phase"]]

Number of undetermined phases: 57


,SubjectKeyRadiology,ExamDate,ProtocolName,contrast,phase_timing,DICOM_phase
1691,NKI-d23231-00-0164,2017-10-02,Lever 1.0 Vol.,0,0.0,NaN
1707,NKI-d23231-00-0180,2017-08-29,Lever 1.0 Vol.,0,0.0,NaN
1876,NKI-d23231-00-0151,2004-11-17,Lever blanco 1.5 B25f,0,0.0,NaN
2019,NKI-d23231-00-0117,2006-11-29,Thorax 1.5 B45f,0,0.0,NaN
2116,NKI-d23231-00-0145,2006-04-19,Thorax 1.5 B45f,0,0.0,NaN
2256,NKI-d23231-00-0155,2007-07-12,Thorax 1.5 B45f,0,0.0,NaN
2280,NKI-d23231-00-0161,2009-12-10,Thorax 1.5 B45f,0,0.0,NaN
2285,NKI-d23231-00-0162,2006-08-09,Abdomen 1.5 B25f,0,0.0,NaN
2347,NKI-d23231-00-0170,2006-04-05,Abdomen 1.5 B25f,0,Just Right,NaN
2558,NKI-d23231-00-0193,2016-10-12,Body 1.0 CE Vol.,0,Too Early,NaN


In [49]:
# ASK KALINA IF LATE PHASE CAN BE REMOVED
data[(data["contrast"] == "Late Phase") | (data["DICOM_phase"].isin(["Excretory", "Late Phase"]))][["ProtocolName", "contrast", "phase_timing", "DICOM_phase"]]

,ProtocolName,contrast,phase_timing,DICOM_phase
303,PORTAAL 3MM.,Late Phase,Just Right,Portal
460,ThxAbd 1.5 B25f,Late Phase,Too Late,NaN
560,Lever 5 min. 1.5 B25f,Late Phase,Too Late,Late Phase
664,lever laat 3.0 B30s,Late Phase,Too Late,NaN
938,Abdomen +C 2mm,Late Phase,Just Right,NaN
1019,Lever 5 min. 1.5 B25f,Late Phase,Too Late,Late Phase
1079,Uitscheiding 1.5 B25f,Late Phase,Too Late,Excretory
1474,Lever +C 1.5 B25f,Late Phase,Too Late,NaN
1568,Lever 5 min. 1.5 B25f,Portal,Just Right,Late Phase
1719,Laat Veneus 3.0 B30s,Late Phase,Too Late,Late Venous


### Clearing and Saving

In [50]:
data_cleared = data[data.file.notna()]
print("True phase labels:\n", data_cleared["contrast"].value_counts())
print("\n")
print("True timing labels:\n",data_cleared["phase_timing"].value_counts())
print('\n')

True phase labels:
 contrast
Portal          3706
Arterial        2487
Non-contrast    2122
0                419
Late Phase        15
Name: count, dtype: int64


True timing labels:
 phase_timing
Just Right      5112
Non-contrast    2122
Too Early        916
0.0              416
Too Late         183
Name: count, dtype: int64




In [51]:
# Drop Late Phase
data_cleared = data[~((data["contrast"] == "Late Phase") |
                      (data["DICOM_phase"].isin(["Excretory", "Late Phase"])))]

print("True:\n", data_cleared["contrast"].value_counts())
print("\n")
print("True:\n",data_cleared["phase_timing"].value_counts())
print('\n')

True:
 contrast
Portal          3700
Arterial        2487
Non-contrast    2118
0                403
Name: count, dtype: int64


True:
 phase_timing
Just Right      5103
Non-contrast    2118
Too Early        916
0.0              400
Too Late         171
Name: count, dtype: int64




In [52]:
data_cleared[(data_cleared["contrast"] == "Late Phase") | (data_cleared["DICOM_phase"].isin(["Excretory", "Late Phase"]))][["ProtocolName", "contrast", "phase_timing", "DICOM_phase"]]


,ProtocolName,contrast,phase_timing,DICOM_phase


In [57]:
data_cleared[(data_cleared['contrast'] != "Non-contrast") & (data_cleared.DICOM_phase == 'Non-contrast')][['contrast', 'DICOM_phase', 'ProtocolName']]

,contrast,DICOM_phase,ProtocolName


In [54]:
print("Length of data_cleared:", len(data_cleared))
display(data_cleared.contrast.value_counts())

Length of data_cleared: 8708


contrast
Portal          3700
Arterial        2487
Non-contrast    2118
0                403
Name: count, dtype: int64

In [55]:
data_cleared.to_csv("/projects/net_contrast_classification/contrast_phase/data/cleaned_data.csv", index=False)

In [56]:
data_cleared

,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,phase_timing,is_lesionfree,...,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness,AcquisitionTime_sec,DICOM_phase
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,DICOM-batch1-kalina01,NET_0000_0000.nii.gz,NKI-d23231-00-0063,2012-04-16,NET_0000_0000.nii.gz,Partially,Arterial,Too Early,No,...,150.0,75.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,40231.70000,Arterial
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,DICOM-batch1-kalina01,NET_0001_0000.nii.gz,NKI-d23231-00-0070,2015-08-21,NET_0001_0000.nii.gz,Yes,Non-contrast,Non-contrast,No,...,76.0,38.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,LEVER,NaN,1.0,30853.90000,Non-contrast
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,DICOM-batch1-kalina01,NET_0002_0000.nii.gz,NKI-d23231-00-0036,2014-02-18,NET_0002_0000.nii.gz,Yes,Portal,Just Right,Yes,...,75.0,37.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,32599.60000,NaN
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,DICOM-batch1-kalina01,NET_0003_0000.nii.gz,NKI-d23231-00-0071,2013-09-24,NET_0003_0000.nii.gz,Yes,Arterial,Just Right,Yes,...,168.0,84.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,45013.30692,NaN
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,DICOM-batch1-kalina01,NET_0004_0000.nii.gz,NKI-d23231-00-0077,2014-12-22,NET_0004_0000.nii.gz,Yes,Portal,Just Right,No,...,119.0,74.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,51587.10418,Portal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8744,../DICOM-batch3\NKI-d23231-00-0888\20220420 CT...,DICOM-batch3-kalina99,NET_5827_0000.nii.gz,NKI-d23231-00-0888,2022-04-20,NET_5827_0000.nii.gz,Yes,Portal,Just Right,No,...,158.0,98.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,54796.11932,Portal
8745,../DICOM-batch3\NKI-d23231-00-0888\20230424 CT...,DICOM-batch3-kalina99,NET_5828_0000.nii.gz,NKI-d23231-00-0888,2023-04-24,NET_5828_0000.nii.gz,Yes,Arterial,Just Right,Yes,...,316.0,175.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,43545.56100,Arterial
8746,../DICOM-batch3\NKI-d23231-00-0888\20230424 CT...,DICOM-batch3-kalina99,NET_5829_0000.nii.gz,NKI-d23231-00-0888,2023-04-24,NET_5829_0000.nii.gz,Yes,Portal,Just Right,No,...,304.0,168.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,43580.13800,NaN
8747,../DICOM-batch3\NKI-d23231-00-0888\20230801 CT...,DICOM-batch3-kalina99,NET_5830_0000.nii.gz,NKI-d23231-00-0888,2023-08-01,NET_5830_0000.nii.gz,Yes,Arterial,Just Right,Yes,...,167.0,92.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,42200.65000,Arterial
